# 04 — Analista conversacional — LockBit Panel

Interfaz de consulta en lenguaje natural sobre la base de datos.  
El LLM recibe el esquema de los DataFrames, genera código pandas para responder  
y el notebook lo ejecuta mostrando código + resultado.

**Uso**: edita la celda de pregunta y ejecuta. Puedes hacer preguntas en español o inglés.

## Setup — ejecutar una vez al abrir el notebook

In [1]:
# "re" es el módulo de expresiones regulares. Lo usamos para extraer el bloque de código
# Python que el LLM genera dentro de su respuesta (entre ``` y ```).
import re

# "json" para manejar datos estructurados (aunque en este notebook lo importamos por precaución)
import json

# "pandas" para trabajar con las tablas de la base de datos de LockBit
import pandas as pd

# "ollama" para enviar preguntas al modelo de lenguaje local y recibir respuestas
import ollama

# "Path" para manejar rutas de archivos de forma cómoda
from pathlib import Path

# "display" y "Markdown" de IPython son las herramientas que permiten mostrar
# texto formateado (negrita, código coloreado) y tablas directamente en el notebook.
# Son exclusivas de Jupyter y hacen la salida mucho más legible que print().
from IPython.display import display, Markdown

# Ruta de la carpeta con los datos procesados
PROCESSED = Path('../data_Vruto/LockBit')

# Nombre del modelo de lenguaje que actuará como analista.
# Este modelo recibirá preguntas en español/inglés y generará código pandas para responderlas.
MODEL     = 'qwen3.5'

# Cargamos todas las tablas disponibles de la base de datos de LockBit.
# Usamos 'chats_classified' (con columna 'phase') si existe del notebook 02;
# si no, usamos los chats básicos.
users    = pd.read_parquet(PROCESSED / 'users.parquet')
clients  = pd.read_parquet(PROCESSED / 'clients.parquet')
builds   = pd.read_parquet(PROCESSED / 'builds.parquet')
invites  = pd.read_parquet(PROCESSED / 'invites.parquet')
btc      = pd.read_parquet(PROCESSED / 'btc_addresses.parquet')

chats_path = PROCESSED / 'chats_classified.parquet'
chats = pd.read_parquet(chats_path if chats_path.exists() else PROCESSED / 'chats.parquet')

# Creamos un diccionario con todos los DataFrames disponibles.
# Este diccionario se usará de dos formas:
# 1. Para construir el contexto de esquema que enviamos al LLM
# 2. Como el "namespace" (espacio de nombres) donde se ejecutará el código generado
DATAFRAMES = {
    'users': users,
    'clients': clients,
    'chats': chats,
    'builds': builds,
    'invites': invites,
    'btc_addresses': btc,
}

# Mostramos un resumen de los datos disponibles para el analista
print('Tablas cargadas:')
for name, df in DATAFRAMES.items():
    print(f'  {name:20s} {len(df):6,} filas x {len(df.columns)} cols')

Tablas cargadas:
  users                    75 filas x 28 cols
  clients                 246 filas x 26 cols
  chats                 3,977 filas x 16 cols
  builds                1,183 filas x 18 cols
  invites               3,693 filas x 8 cols
  btc_addresses        59,975 filas x 5 cols


In [2]:
def _build_schema_context() -> str:
    """
    Construye un texto descriptivo de todas las tablas disponibles para enviarlo al LLM.

    Para que el LLM pueda escribir código pandas correcto, necesita saber:
    - Qué tablas existen y cuántas filas tienen
    - Qué columnas tiene cada tabla y de qué tipo son
    - Ejemplos de valores reales para entender qué contiene cada columna

    Esta función genera automáticamente ese texto a partir de los DataFrames cargados.

    Devuelve:
        str: Un bloque de texto con el esquema completo de todas las tablas.
    """
    lines = ["Available pandas DataFrames (LockBit ransomware panel DB, Dec 2024–Apr 2025):\n"]

    for name, df in DATAFRAMES.items():
        # Nombre de la tabla y número de filas
        lines.append(f"  {name} ({len(df):,} rows)")

        # Lista de columnas con su tipo de dato
        # df[c].dtype devuelve el tipo de la columna: int64, float64, object (texto), datetime64, etc.
        col_info = ', '.join(
            f"{c} [{df[c].dtype}]" for c in df.columns
        )
        lines.append(f"    columns: {col_info}")

        # Añadimos una fila de ejemplo para que el LLM entienda qué valores son reales
        # dropna(how='all') descarta filas donde todos los valores son nulos
        sample = df.dropna(how='all').head(1)
        if len(sample):
            # Solo mostramos las primeras 8 columnas para no saturar el contexto del LLM
            for col in sample.columns[:8]:
                val = sample[col].iloc[0]
                # Limitamos a 60 caracteres y quitamos saltos de línea para que quede en una línea
                val_str = str(val)[:60].replace('\n', ' ')
                lines.append(f"    {col}: {val_str}")
        lines.append('')  # Línea en blanco entre tablas

    return '\n'.join(lines)


# Generamos el contexto de esquema una sola vez y lo guardamos en una constante.
# Este texto se incluirá en cada pregunta que enviemos al LLM.
SCHEMA_CONTEXT = _build_schema_context()

# El system prompt es la "carta de presentación" que recibe el LLM al inicio de la sesión.
# Le dice quién es, qué datos tiene disponibles y cómo debe responder.
# Incluye el esquema completo de las tablas, relaciones entre ellas y reglas de formato.
# Cuanto más preciso y completo sea este prompt, mejores serán las respuestas del LLM.
SYSTEM_PROMPT = f"""You are a forensic data analyst working with the leaked LockBit ransomware panel database.
Answer questions by writing Python/pandas code that queries the available DataFrames.

{SCHEMA_CONTEXT}
Key facts:
- `clients` = compromised victim companies. `paid_commission=1` means they paid the ransom.
- `chats` = negotiations between LockBit operators (flag=1) and victims (flag=0).
- `chats.phase` (if present) = negotiation phase classified by LLM.
- `users` = LockBit operators/affiliates. `is_admin=1` = admin.
- `builds` = malware builds created by operators. `type` = LockBit variant (25/30/40/46/50).
- `invites` = affiliate recruitment codes. `status=50` = accepted.
- Timestamps are UTC. Unix timestamps in `clients.date_first`, `clients.date_last`, `builds.date`.
- `clients.advid` and `chats.advid` link to `users.id`.
- `clients.build_id` links to `builds.id`.

Rules:
- Write clean, runnable Python using pandas. Do NOT import anything — all libraries are already in scope.
- Store your final answer in a variable called `result`.
- `result` can be a DataFrame, Series, scalar, dict, or string.
- Add brief inline comments.
- If the question cannot be answered with the available data, say so clearly.

Always wrap your code in a ```python ... ``` block."""

# Lista que almacena el historial de la conversación actual.
# Cada entrada es un diccionario con 'role' ('user' o 'assistant') y 'content' (el texto).
# Acumular el historial permite hacer preguntas de seguimiento donde el LLM recuerda
# el contexto de preguntas anteriores ("¿Y cuántos builds tiene ese operador?").
# La variable usa _ al inicio por convención: indica que es interna y no debe modificarse directamente.
_history = []

print('Sistema de consulta listo.')
print(f'Modelo: {MODEL}')
print('Usa ask("tu pregunta") para consultar.')

Sistema de consulta listo.
Modelo: qwen3.5
Usa ask("tu pregunta") para consultar.


In [3]:
def ask(question: str, show_code: bool = True, memory: bool = True):
    """
    El corazón del analista conversacional: permite "chatear con los datos" en lenguaje natural.

    ¿Qué significa "chatear con los datos"?
    En lugar de aprender pandas para escribir consultas complejas, el usuario hace preguntas
    normales en español o inglés ("¿Qué operador tiene más víctimas?") y el sistema:
      1. Envía la pregunta a un LLM con el contexto de los datos disponibles
      2. El LLM genera código Python/pandas para responder esa pregunta
      3. El notebook ejecuta ese código automáticamente
      4. Se muestra el código Y el resultado para que el usuario pueda aprender y verificar

    Este patrón se llama "Text-to-Code" o "NL2Code" (Natural Language to Code).
    Es una de las aplicaciones más prácticas de los LLMs en análisis de datos.

    Parámetros:
        question (str): La pregunta en lenguaje natural. Puede ser en español o inglés.
                        Ejemplo: "¿Cuántas víctimas hay por operador?"
        show_code (bool): Si True (por defecto), muestra el código generado antes del resultado.
                          Útil para aprender qué código pandas responde a cada pregunta.
                          Ponlo en False si solo te interesa el resultado.
        memory (bool): Si True (por defecto), incluye el historial de preguntas anteriores
                       en el contexto. Esto permite preguntas encadenadas como:
                       ask("¿Quién tiene más víctimas?")
                       ask("¿Cuántos builds tiene ese operador?")  # el LLM recuerda el contexto
                       Ponlo en False si quieres una pregunta completamente independiente.
    """
    global _history  # Usamos 'global' para poder modificar la lista _history desde dentro de la función

    # Construimos la lista de mensajes que enviaremos al LLM.
    # Siempre empezamos con el system prompt (instrucciones generales + esquema de datos).
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]

    # Si la memoria está activada, añadimos el historial de preguntas y respuestas anteriores.
    # Esto es lo que permite al LLM "recordar" el contexto de preguntas anteriores.
    if memory:
        messages += _history

    # Añadimos la nueva pregunta del usuario al final de la conversación
    messages.append({'role': 'user', 'content': question})

    # Enviamos toda la conversación al LLM y esperamos su respuesta
    resp = ollama.chat(
        model=MODEL,
        messages=messages,
        # temperature=0.1 produce respuestas más consistentes y precisas (menos "creativas")
        # ideal para generación de código donde la precisión es crítica
        options={'temperature': 0.1}
    )
    answer = resp.message.content.strip()

    # Actualizamos el historial con la pregunta y la respuesta para la próxima llamada
    if memory:
        _history.append({'role': 'user', 'content': question})
        _history.append({'role': 'assistant', 'content': answer})

        # Limitamos el historial a los últimos 20 mensajes (10 pares pregunta-respuesta)
        # para no saturar la ventana de contexto del LLM con conversaciones muy largas
        if len(_history) > 20:
            _history = _history[-20:]

    # Buscamos el bloque de código Python en la respuesta del LLM.
    # El LLM debe envolver su código entre ```python y ``` según las instrucciones del system prompt.
    # re.search() busca el patrón en el texto; re.DOTALL hace que '.' coincida también con '\n'
    code_match = re.search(r'```python\n(.*?)```', answer, re.DOTALL)

    if code_match:
        # Extraemos el código del bloque ``` ``` y eliminamos espacios sobrantes
        code = code_match.group(1).strip()

        if show_code:
            # Mostramos el código generado con formato Markdown (con resaltado de sintaxis)
            # Esto ayuda al usuario a entender qué hizo el LLM y a aprender pandas
            display(Markdown(f'**Código generado:**\n```python\n{code}\n```'))

        # Ejecutamos el código generado en un namespace aislado.
        # El namespace contiene todos los DataFrames + pandas como 'pd'.
        # Esto simula el entorno del notebook: el código puede usar 'clients', 'chats', etc.
        # sin necesidad de importarlos porque ya están en el namespace.
        # ** convierte el diccionario DATAFRAMES en argumentos (desempaquetado)
        namespace = {**DATAFRAMES, 'pd': pd, 'result': None}

        try:
            # exec() ejecuta el código como si fuera una celda del notebook
            exec(code, namespace)

            # El LLM debe guardar su respuesta en una variable llamada 'result'
            # según las reglas del system prompt. La recuperamos del namespace.
            result = namespace.get('result')

            display(Markdown('**Resultado:**'))
            if result is not None:
                # display() muestra el resultado con formato visual:
                # - Si es un DataFrame, lo muestra como tabla HTML
                # - Si es un número, lo muestra directamente
                # - Si es texto, lo muestra como texto
                display(result)
            else:
                print('(sin resultado — revisa el código generado)')

        except Exception as e:
            # Si el código generado tiene un error (sintaxis, columna inexistente, etc.)
            # mostramos el error y la respuesta original del LLM para debugging
            print(f'Error al ejecutar el código: {e}')
            display(Markdown('**Respuesta sin código:**'))
            display(Markdown(answer))
    else:
        # Si el LLM respondió sin código (por ejemplo, diciendo que no puede responder),
        # mostramos su respuesta directamente en formato Markdown
        display(Markdown(answer))


def reset_memory():
    """
    Borra el historial de conversación para empezar una nueva sesión de preguntas.

    El historial permite hacer preguntas de seguimiento ("¿Y ese operador?"),
    pero también puede confundir al LLM si cambiamos de tema radicalmente.
    En ese caso, llama a reset_memory() antes de empezar la nueva línea de preguntas.

    Ejemplo de uso:
        ask("¿Quién tiene más víctimas?")
        ask("¿Y cuántos mensajes de chat tiene?")  # sigue el hilo del anterior
        reset_memory()
        ask("¿Qué builds se crearon en enero?")   # nueva conversación independiente
    """
    global _history
    _history = []  # Vaciamos la lista del historial
    print('Historial borrado.')

---
## Preguntas de ejemplo

Edita cualquier celda y ejecuta. Puedes añadir celdas nuevas con `B`.

In [4]:
# Pregunta de ejemplo 1: estadísticas básicas de víctimas y pagos.
# El LLM generará código pandas que cuente las filas de 'clients' y
# cuántas tienen paid_commission == 1.
ask("¿Cuántas víctimas hay en total y cuántas pagaron el rescate?")

**Código generado:**
```python
import pandas as pd

# Count total victims and those who paid the ransom
total_victims = len(clients)
paid_victims = len(clients[clients['paid_commission'] == 1])

# Store result as a dictionary with clear labels
result = {
    'total_victims': total_victims,
    'paid_victims': paid_victims,
    'paid_percentage': round(paid_victims / total_victims * 100, 2) if total_victims > 0 else 0
}

# Display the result
print(f"Total víctimas: {total_victims}")
print(f"Víctimas que pagaron rescate: {paid_victims}")
print(f"Porcentaje que pagó: {result['paid_percentage']}%")
```

Total víctimas: 246
Víctimas que pagaron rescate: 7
Porcentaje que pagó: 2.85%


**Resultado:**

{'total_victims': 246, 'paid_victims': 7, 'paid_percentage': 2.85}

In [5]:
# Pregunta de ejemplo 2: el LLM deberá unir la tabla 'clients' con 'users'
# usando clients.advid == users.id para poder mostrar el nombre de login
# en lugar del ID numérico del operador.
ask("¿Qué operador tiene más víctimas asignadas? Muestra su login y número de víctimas")

**Código generado:**
```python
import pandas as pd

# Join clients with users to count victims per operator
# clients.advid links to users.id
merged = clients.merge(users[['id', 'login']], left_on='advid', right_on='id', how='left')

# Count victims per operator (excluding rows where login is NaN)
operator_counts = merged.groupby('login').size().reset_index(name='victim_count')

# Sort by victim count descending and get the top operator
top_operator = operator_counts.sort_values('victim_count', ascending=False).head(1)

# Store result
result = {
    'login': top_operator['login'].values[0] if len(top_operator) > 0 else None,
    'victim_count': int(top_operator['victim_count'].values[0]) if len(top_operator) > 0 else 0
}

# Display the result
print(f"Operador con más víctimas: {result['login']}")
print(f"Número de víctimas: {result['victim_count']}")
```

Operador con más víctimas: Christopher
Número de víctimas: 44


**Resultado:**

{'login': 'Christopher', 'victim_count': 44}

In [6]:
# Pregunta de ejemplo 3: análisis temporal. El LLM deberá agrupar por mes
# usando resample() o groupby() sobre la columna de fecha 'date_first'.
ask("¿Cuál fue el mes con más nuevas víctimas comprometidas?")

**Código generado:**
```python
import pandas as pd

# Extract month from date_first column and count victims per month
# date_first is when the victim was first compromised
monthly_victims = clients.groupby(clients['date_first'].dt.to_period('M')).size().reset_index(name='victim_count')

# Sort by victim count descending and get the top month
top_month = monthly_victims.sort_values('victim_count', ascending=False).head(1)

# Store result
result = {
    'month': top_month['date_first'].dt.strftime('%Y-%m').values[0] if len(top_month) > 0 else None,
    'victim_count': int(top_month['victim_count'].values[0]) if len(top_month) > 0 else 0
}

# Display the result
print(f"Mes con más nuevas víctimas: {result['month']}")
print(f"Número de víctimas: {result['victim_count']}")
```

Mes con más nuevas víctimas: 2025-01
Número de víctimas: 84


<string>:5: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.


**Resultado:**

{'month': '2025-01', 'victim_count': 84}

In [7]:
# Pregunta de ejemplo 4: análisis complejo que requiere combinar tablas.
# El LLM necesitará filtrar víctimas que pagaron, unirlas con usuarios
# y calcular la diferencia de fechas entre primer contacto y pago.
ask("Muestra las víctimas que pagaron: su ID, operador asignado, y días que tardaron en pagar desde el primer contacto")

**Código generado:**
```python
import pandas as pd

# Filter victims who paid the ransom
paid_clients = clients[clients['paid_commission'] == 1]

# Join with users to get operator login (advid links to users.id)
paid_clients = paid_clients.merge(users[['id', 'login']], left_on='advid', right_on='id', how='left')

# Calculate days from first contact to last contact (approximate payment date)
# date_first = first contact, date_last = last contact (likely after payment)
paid_clients['days_to_pay'] = (paid_clients['date_last'] - paid_clients['date_first']).dt.days

# Select and format the required columns
result = paid_clients[['id', 'login', 'days_to_pay']].copy()

# Sort by days_to_pay for better readability
result = result.sort_values('days_to_pay')

# Display the result
print(result)
```

Error al ejecutar el código: "['id'] not in index"


**Respuesta sin código:**

```python
import pandas as pd

# Filter victims who paid the ransom
paid_clients = clients[clients['paid_commission'] == 1]

# Join with users to get operator login (advid links to users.id)
paid_clients = paid_clients.merge(users[['id', 'login']], left_on='advid', right_on='id', how='left')

# Calculate days from first contact to last contact (approximate payment date)
# date_first = first contact, date_last = last contact (likely after payment)
paid_clients['days_to_pay'] = (paid_clients['date_last'] - paid_clients['date_first']).dt.days

# Select and format the required columns
result = paid_clients[['id', 'login', 'days_to_pay']].copy()

# Sort by days_to_pay for better readability
result = result.sort_values('days_to_pay')

# Display the result
print(result)
```

In [8]:
# Pregunta de ejemplo 5: clasificación/ranking de operadores por builds creados.
# El LLM agrupará 'builds' por 'userid', contará, unirá con 'users' para obtener
# el login y mostrará los 10 más activos.
ask("¿Cuántos builds de malware creó cada operador? Muestra los 10 más activos con su login")

**Código generado:**
```python
import pandas as pd

# Join builds with users to get operator login (builds.userid links to users.id)
builds_with_users = builds.merge(users[['id', 'login']], left_on='userid', right_on='id', how='left')

# Count builds per operator
build_counts = builds_with_users.groupby('login').size().reset_index(name='build_count')

# Sort by build count descending and get top 10
top_builders = build_counts.sort_values('build_count', ascending=False).head(10)

# Store result
result = top_builders[['login', 'build_count']].copy()

# Display the result
print("Top 10 operadores más activos en creación de builds:")
print(result)
```

Top 10 operadores más activos en creación de builds:
            login  build_count
17     JamesCraig          170
28           Swan          147
36  btcdrugdealer           80
16       Iofikdis           79
50   umarbishop47           74
10    Christopher           70
46      matrix777           56
25      PiotrBond           53
2         Anon666           32
27   RiccardoBond           30


**Resultado:**

,login,build_count
17,JamesCraig,170
28,Swan,147
36,btcdrugdealer,80
16,Iofikdis,79
50,umarbishop47,74
10,Christopher,70
46,matrix777,56
25,PiotrBond,53
2,Anon666,32
27,RiccardoBond,30


In [9]:
# Pregunta de ejemplo 6: pregunta con dos partes: cuál es el tipo más frecuente
# Y si eso ha cambiado con el tiempo (evolución temporal por versión).
# El LLM deberá usar value_counts() y probablemente resample() por semana o mes.
ask("¿Cuál es la variante de LockBit más usada (campo type)? ¿Ha cambiado con el tiempo?")

**Código generado:**
```python
import pandas as pd

# Count builds per variant type
variant_counts = builds.groupby('type').size().reset_index(name='build_count')

# Sort by build count descending
variant_counts = variant_counts.sort_values('build_count', ascending=False)

# Get the most used variant
most_used_variant = variant_counts.iloc[0]

# Analyze changes over time - group by year and month
builds_with_date = builds.copy()
builds_with_date['date'] = pd.to_datetime(builds_with_date['date'])
builds_with_date['year'] = builds_with_date['date'].dt.year
builds_with_date['month'] = builds_with_date['date'].dt.to_period('M')

# Count builds per type per month
variant_over_time = builds_with_date.groupby(['type', 'month']).size().reset_index(name='build_count')

# Store result
result = {
    'most_used_variant': int(most_used_variant['type'].values[0]) if len(most_used_variant) > 0 else None,
    'total_builds': int(most_used_variant['build_count'].values[0]) if len(most_used_variant) > 0 else 0,
    'percentage': round(most_used_variant['build_count'].values[0] / len(builds) * 100, 2) if len(builds) > 0 else 0,
    'has_changed': len(variant_over_time) > 1 and not variant_over_time['type'].nunique() == 1
}

# Display the result
print(f"Variante más usada: {result['most_used_variant']}")
print(f"Total de builds: {result['total_builds']}")
print(f"Porcentaje: {result['percentage']}%")
print(f"Ha cambiado con el tiempo: {result['has_changed']}")

# Show distribution over time if there are multiple variants
if len(variant_over_time) > 1:
    print("\nDistribución de variantes por mes (últimos 6 meses):")
    recent = variant_over_time.tail(6)
    print(recent)
```

Error al ejecutar el código: 'numpy.int64' object has no attribute 'values'


<string>:16: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.


**Respuesta sin código:**

```python
import pandas as pd

# Count builds per variant type
variant_counts = builds.groupby('type').size().reset_index(name='build_count')

# Sort by build count descending
variant_counts = variant_counts.sort_values('build_count', ascending=False)

# Get the most used variant
most_used_variant = variant_counts.iloc[0]

# Analyze changes over time - group by year and month
builds_with_date = builds.copy()
builds_with_date['date'] = pd.to_datetime(builds_with_date['date'])
builds_with_date['year'] = builds_with_date['date'].dt.year
builds_with_date['month'] = builds_with_date['date'].dt.to_period('M')

# Count builds per type per month
variant_over_time = builds_with_date.groupby(['type', 'month']).size().reset_index(name='build_count')

# Store result
result = {
    'most_used_variant': int(most_used_variant['type'].values[0]) if len(most_used_variant) > 0 else None,
    'total_builds': int(most_used_variant['build_count'].values[0]) if len(most_used_variant) > 0 else 0,
    'percentage': round(most_used_variant['build_count'].values[0] / len(builds) * 100, 2) if len(builds) > 0 else 0,
    'has_changed': len(variant_over_time) > 1 and not variant_over_time['type'].nunique() == 1
}

# Display the result
print(f"Variante más usada: {result['most_used_variant']}")
print(f"Total de builds: {result['total_builds']}")
print(f"Porcentaje: {result['percentage']}%")
print(f"Ha cambiado con el tiempo: {result['has_changed']}")

# Show distribution over time if there are multiple variants
if len(variant_over_time) > 1:
    print("\nDistribución de variantes por mes (últimos 6 meses):")
    recent = variant_over_time.tail(6)
    print(recent)
```

In [10]:
# Pregunta de ejemplo 7: estadísticas de duración de negociaciones.
# El LLM agrupará 'chats' por 'clientid', calculará min y max de 'created_at'
# por grupo, restará las fechas y convertirá a horas. Luego calculará la mediana.
ask("De las negociaciones de chat, ¿cuánto tiempo median (en horas) entre el primer y último mensaje por víctima?")

**Código generado:**
```python
import pandas as pd

# Filter chats that are not from operators (flag=0 = victim messages)
victim_chats = chats[chats['flag'] == 0]

# Group by clientid (victim) and find first and last message dates
chat_times = victim_chats.groupby('clientid').agg(
    first_message=('date', 'min'),
    last_message=('date', 'max')
).reset_index()

# Calculate time difference in hours
chat_times['hours_between'] = (chat_times['last_message'] - chat_times['first_message']).dt.total_seconds() / 3600

# Store result
result = chat_times[['clientid', 'first_message', 'last_message', 'hours_between']].copy()

# Sort by hours_between descending to show longest negotiations first
result = result.sort_values('hours_between', ascending=False)

# Display the result
print("Negociaciones de chat por víctima (tiempo entre primer y último mensaje):")
print(result.head(20))  # Show top 20 longest negotiations
print(f"\nTotal de víctimas con negociaciones: {len(result)}")
print(f"Tiempo promedio de negociación: {result['hours_between'].mean():.2f} horas")
print(f"Negociación más larga: {result['hours_between'].max():.2f} horas")
```

Negociaciones de chat por víctima (tiempo entre primer y último mensaje):
     clientid             first_message              last_message  \
0          24 2024-12-19 04:45:24+00:00 2025-04-21 13:40:38+00:00   
52        126 2025-01-26 09:27:47+00:00 2025-04-02 17:03:12+00:00   
83        157 2025-02-12 06:56:27+00:00 2025-04-12 10:00:12+00:00   
135       209 2025-03-22 13:07:03+00:00 2025-04-26 21:18:48+00:00   
126       200 2025-03-17 06:50:50+00:00 2025-04-21 10:41:38+00:00   
25         59 2025-01-05 09:21:15+00:00 2025-02-06 15:39:20+00:00   
139       213 2025-03-26 01:07:49+00:00 2025-04-25 23:43:35+00:00   
14         47 2024-12-27 12:53:43+00:00 2025-01-24 12:29:07+00:00   
160       234 2025-03-30 13:06:56+00:00 2025-04-24 10:05:12+00:00   
167       241 2025-04-03 16:09:27+00:00 2025-04-27 15:14:13+00:00   
161       235 2025-03-31 01:36:02+00:00 2025-04-23 23:45:58+00:00   
1          25 2024-12-19 22:02:40+00:00 2025-01-11 06:14:58+00:00   
80        154 2025-02-11 05:0

**Resultado:**

,clientid,first_message,last_message,hours_between
0,24,2024-12-19 04:45:24+00:00,2025-04-21 13:40:38+00:00,2960.920556
52,126,2025-01-26 09:27:47+00:00,2025-04-02 17:03:12+00:00,1591.590278
83,157,2025-02-12 06:56:27+00:00,2025-04-12 10:00:12+00:00,1419.062500
135,209,2025-03-22 13:07:03+00:00,2025-04-26 21:18:48+00:00,848.195833
126,200,2025-03-17 06:50:50+00:00,2025-04-21 10:41:38+00:00,843.846667
...,...,...,...,...
182,256,2025-04-16 08:57:20+00:00,2025-04-16 08:57:20+00:00,0.000000
188,262,2025-04-18 08:27:22+00:00,2025-04-18 08:27:22+00:00,0.000000
186,260,2025-04-17 10:32:54+00:00,2025-04-17 10:32:54+00:00,0.000000
195,269,2025-04-22 10:42:37+00:00,2025-04-22 10:42:37+00:00,0.000000


---
## Tu turno — edita esta celda con tu pregunta

In [11]:
# Pregunta de ejemplo 8: exploración del esquema de datos.
# Útil para que el LLM mismo describa qué hay disponible cuando el usuario
# no sabe qué preguntar. El LLM tiene acceso al esquema en el system prompt.
ask("¿Cuál es la estructura de la base de datos?")

**Código generado:**
```python
import pandas as pd

# Database structure overview
db_structure = {
    'users': {
        'rows': 75,
        'description': 'Operadores/afiliados de LockBit',
        'columns': ['id', 'parentid', 'login', 'password', 'is_admin', 'level', 'session_id', 
                   'linesxi_on', 'reg_date', 'last_online', 'paused', 'builders_settings', 
                   'notifications', 'notify_trial', 'negotiations', 'show_extra_info', 
                   'keep_messages_unread', 'toxid', 'toxdata', 'ips', 'permissions', 
                   'paranoid_mode', 'created_at', 'updated_at', 'tag', 'invite_id', 
                   'last_online_dt', 'reg_date_dt']
    },
    'clients': {
        'rows': 246,
        'description': 'Empresas víctimas comprometidas',
        'columns': ['id', 'important', 'advid', 'master_pubkey', 'session_key', 
                   'paid_commission', 'trial_done', 'decrypt_done', 'decrypt_2_done', 
                   'decrypt_3_done', 'decrypt_done_at', 'decrypt_2_done_at', 'decrypt_3_done_at', 
                   'chat_status', 'can_chat', 'banned', 'banned_at', 'banned_reason', 
                   'date_first', 'date_last', 'date_first_dt', 'date_last_dt', 
                   'date_first', 'date_last', 'date_first_dt', 'date_last_dt']
    },
    'chats': {
        'rows': 3977,
        'description': 'Negociaciones entre operadores y víctimas',
        'columns': ['id', 'clientid', 'userid', 'date', 'flag', 'message', 
                   'date_dt', 'date', 'date_dt']
    },
    'builds': {
        'rows': 1183,
        'description': 'Builds de malware creados por operadores',
        'columns': ['id', 'userid', 'date', 'type', 'version', 'build', 'build_date', 
                   'build_date_dt', 'build_date', 'build_date_dt']
    },
    'invites': {
        'rows': 3693,
        'description': 'Códigos de reclutamiento de afiliados',
        'columns': ['id', 'userid', 'date', 'code', 'code_dt', 'date', 'date_dt']
    },
    'btc_addresses': {
        'rows': 59975,
        'description': 'Direcciones de Bitcoin relacionadas',
        'columns': ['id', 'clientid', 'date', 'address', 'amount', 'date_dt', 
                   'date', 'date_dt']
    }
}

# Display database structure
print("=" * 80)
print("ESTRUCTURA DE LA BASE DE DATOS")
print("=" * 80)

for table_name, info in db_structure.items():
    print(f"\n📊 Tabla: {table_name.upper()}")
    print(f"   Filas: {info['rows']}")
    print(f"   Descripción: {info['description']}")
    print(f"   Columnas ({len(info['columns'])}):")
    for col in info['columns']:
        print(f"      - {col}")

print("\n" + "=" * 80)
print("RESUMEN DE DATOS")
print("=" * 80)

# Show sample data from each table
print("\n📋 Muestra de datos por tabla:")

for table_name, info in db_structure.items():
    if table_name in ['users', 'clients', 'chats', 'builds', 'invites', 'btc_addresses']:
        df = globals()[table_name]
        print(f"\n{table_name.upper()} (primeras 5 filas):")
        print(df.head())
```

ESTRUCTURA DE LA BASE DE DATOS

📊 Tabla: USERS
   Filas: 75
   Descripción: Operadores/afiliados de LockBit
   Columnas (28):
      - id
      - parentid
      - login
      - password
      - is_admin
      - level
      - session_id
      - linesxi_on
      - reg_date
      - last_online
      - paused
      - builders_settings
      - notifications
      - notify_trial
      - negotiations
      - show_extra_info
      - keep_messages_unread
      - toxid
      - toxdata
      - ips
      - permissions
      - paranoid_mode
      - created_at
      - updated_at
      - tag
      - invite_id
      - last_online_dt
      - reg_date_dt

📊 Tabla: CLIENTS
   Filas: 246
   Descripción: Empresas víctimas comprometidas
   Columnas (26):
      - id
      - important
      - advid
      - master_pubkey
      - session_key
      - paid_commission
      - trial_done
      - decrypt_done
      - decrypt_2_done
      - decrypt_3_done
      - decrypt_done_at
      - decrypt_2_done_at
      - decry

**Resultado:**

(sin resultado — revisa el código generado)


---
## Preguntas de seguimiento (el LLM recuerda el contexto de la sesión)

Las preguntas encadenadas funcionan porque el historial se acumula.  
Usa `reset_memory()` para empezar una línea de preguntas nueva.

In [12]:
# Ejemplo de cómo usar el historial para hacer preguntas encadenadas.
# El LLM recuerda qué respondió antes, por lo que "ese operador" se refiere
# al operador mencionado en la respuesta anterior sin necesidad de nombrarlo.

# Puedes descomentar las líneas de abajo para probarlo:

# Primera pregunta: quién tiene más víctimas
# ask("¿Qué operador tiene más víctimas?")

# Segunda pregunta: sigue el hilo sin repetir el nombre del operador
# ask("¿Cuántos de sus builds son de tipo 30?")  # el LLM sabe a qué operador te refieres

# Cuando quieras empezar una nueva línea de preguntas sin relación con la anterior,
# llama a reset_memory() para limpiar el historial y evitar confusiones:
# reset_memory()  # descomenta para borrar el historial